# HOGformer on Kaggle (train / resume)
Session options (right sidebar): **Accelerator = GPU T4 x2** (or P100), **Internet = On**, **Persistence = Files only**.
Run the cells in order. Only edit the variables in cell 4 (RESUME_FROM_BASELINE) and cell 2 (ALLWEATHER_ID).

In [ ]:
# 1) GPU and disk check
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
!df -h /tmp /kaggle/working

In [ ]:
# 2) Code + dependencies
%cd /kaggle/working
!test -d HOGformer_DLCV || git clone -b claude/lucid-fermi-emhnbo https://github.com/adityaj1706/HOGformer_DLCV.git
%cd /kaggle/working/HOGformer_DLCV
!git pull
!git log -1 --oneline
!pip install -q -r settingI/requirements_colab.txt
ALLWEATHER_ID = '1tfeBnjZX1wIhIFPl6HOzzOKOyo0GdGHl'   # replace with the id of YOUR shared copy if this hits the Drive quota

In [ ]:
# 3) Data -> /tmp/data (not /kaggle/working, which is capped and saved as output).
#    /content/data is a symlink so the configs are IDENTICAL to the Colab runs.
!mkdir -p /tmp/data /content
!ln -sfn /tmp/data /content/data
%cd /kaggle/working/HOGformer_DLCV
!SKIP_SNOW=1 ALLWEATHER_ID={ALLWEATHER_ID} bash scripts/download_data.sh /tmp/data

In [ ]:
# 4) OPTIONAL - resume the baseline from the checkpoint you uploaded as a Kaggle dataset.
#    Needs BOTH net_g_<N>.pth and <N>.state with the same N. Skip this cell for the proposed model.
import glob, os, shutil
RESUME_FROM_BASELINE = True
CK = '/kaggle/input/hogformer-baseline-ckpt'
if RESUME_FROM_BASELINE:
    print(glob.glob(CK + '/**/*', recursive=True))
    exp = '/kaggle/working/experiments/Allweather_HOGformer_baseline'
    os.makedirs(exp + '/models', exist_ok=True); os.makedirs(exp + '/training_states', exist_ok=True)
    for f in glob.glob(CK + '/**/net_g_*.pth', recursive=True): shutil.copy(f, exp + '/models/')
    for f in glob.glob(CK + '/**/*.state', recursive=True): shutil.copy(f, exp + '/training_states/')
    print(os.listdir(exp + '/models'), os.listdir(exp + '/training_states'))

In [ ]:
# 5) TRAIN (baseline). Checkpoints go to /kaggle/working/experiments and are kept as notebook output.
%cd /kaggle/working/HOGformer_DLCV/settingI
!ln -sfn /kaggle/working/experiments experiments
!python -u basicsr/train.py -opt Allweather/Options/Allweather_HOGformer_baseline.yml --launcher none 2>&1 | grep --line-buffered -E "iter: +[0-9]*00,|Validation|Resuming|End of training|Error|Traceback"

In [ ]:
# 6) What was saved
!ls -la /kaggle/working/experiments/Allweather_HOGformer_baseline/models | tail -4
!grep -h Validation /kaggle/working/experiments/Allweather_HOGformer_baseline/train_*.log | tail -6